# Two-Scales Volatility and the Noise

The two-scales estimator of [Zhang, Mykland and Aït-Sahalia (2005)](https://doi.org/10.1198/016214505000000169) uses every tick and still removes the bias of microstructure noise. This tutorial computes it with `two_scales_volatility` of HighFrequencyCovariance.jl, rebuilds it by hand from its formula, tries several numbers of sub-grids, and raises the noise to see where sparse sampling fails and two scales hold.

In [ ]:
Pollis.packages("HighFrequencyCovariance", "Plots")

## The Same Simulated Day

A trading day with a tick every second, a daily volatility of 1% and noise of 2 basis points, as in the first tutorial.

In [ ]:
using HighFrequencyCovariance, Dates, Random, Statistics
import Plots

rng = Xoshiro(7)
n = 23400  # a tick every second over a trading day of 6.5 hours
efficient = log(100) .+ cumsum(0.01 / sqrt(n) .* randn(rng, n))  # efficient log price, daily volatility 1%
noise = 0.0002 .* randn(rng, n)  # microstructure noise of 2 basis points
ticks = (time = collect(1.0:n), asset = fill(:SPX, n), logprice = efficient .+ noise)
ts = SortedDataFrame(HighFrequencyCovariance.DataFrame(ticks), :time, :asset, :logprice, Second(1))
span = duration(ts; in_dates_period = false)  # seconds from the first tick to the last

function grid_returns(t, p, every)  # log returns between the last prices at or before each multiple of every seconds
	grid = range(first(t), last(t); step = every)
	return diff([p[searchsortedlast(t, g)] for g in grid])
end

## Two-Scales Volatility in One Call

`two_scales_volatility` returns two `Dict`s by asset: the volatility per square root of a second and the variance of the noise. With no `num_grids` it uses `default_num_grids(ts)` sub-grids.

In [ ]:
vols, noise_var = two_scales_volatility(ts, [:SPX])
(two_scales = vols[:SPX] * sqrt(span), noise_basis_points = 1e4 * sqrt(noise_var[:SPX]), K = default_num_grids(ts), true_volatility = 0.01, true_noise_basis_points = 2.0)

## The Estimator by Hand

The slow scale averages realized variance over K offset sub-grids, each using every K-th tick, so no tick is discarded. The fast scale is realized variance on every tick, which is mostly noise. With n tick returns and n_bar = (n − K + 1) / K returns per sub-grid, the estimator is TSRV = RV_avg − (n_bar / n) × RV_all, and RV_all / (2n) estimates the noise variance. `two_scales_volatility` applies the correction to volatilities instead, the square roots of the realized variances, with the factor 1/K: the average sub-grid volatility minus 1/K times the volatility on every tick, divided by 1 − 1/K. Its value is therefore close to, but not the same as, the formula of the paper.

In [ ]:
function tsrv(p, K)  # the two-scales realized variance of the paper
	nret = length(p) - 1  # tick returns
	RV_all = sum(diff(p) .^ 2)  # fast scale: every tick
	RV_avg = mean(sum(diff(p[k:K:end]) .^ 2) for k in 1:K)  # slow scale: average over K sub-grids
	n_bar = (nret - K + 1) / K
	return (TSRV = RV_avg - (n_bar / nret) * RV_all, RV_avg = RV_avg, noise = RV_all / (2 * nret))
end
K = default_num_grids(ts)
hand = tsrv(ticks.logprice, K)
(by_hand = sqrt(hand.TSRV), package = vols[:SPX] * sqrt(span), slow_scale_alone = sqrt(hand.RV_avg), noise_by_hand_basis_points = 1e4 * sqrt(hand.noise))

## How the Number of Sub-grids Matters

K sets the slow time scale: with ticks one second apart, K = 300 makes each sub-grid return a five-minute return. Too few sub-grids leave a large correction to make; too many leave few returns in each sub-grid.

In [ ]:
Ks = [3, 10, 30, 100, 234, 600, 1500]
estimates = [two_scales_volatility(ts, [:SPX]; num_grids = k)[1][:SPX] * sqrt(span) for k in Ks]
Plots.plot(Ks, 100 .* estimates; xscale = :log10, marker = :circle, xlabel = "sub-grids K", ylabel = "volatility (%)", label = "two-scales", title = "Two-scales volatility against K")
Plots.hline!([1.0]; linestyle = :dash, label = "true volatility")

## More Noise

The same efficient price is now observed with more noise, and K stays at its default. Realized volatility at five minutes rises with the noise, as each five-minute return still holds the noise of two prices. The formula of the paper, on variances, stays close to the true volatility. `two_scales_volatility`, which corrects the volatilities, rises as well when the noise is strong; a larger `num_grids` brings it closer.

In [ ]:
rows = map([0.0, 0.0002, 0.0005, 0.001, 0.002]) do sd
	noisy = (time = ticks.time, asset = ticks.asset, logprice = efficient .+ sd .* randn(Xoshiro(11), n))
	tsn = SortedDataFrame(HighFrequencyCovariance.DataFrame(noisy), :time, :asset, :logprice, Second(1))
	v, w = two_scales_volatility(tsn, [:SPX])
	(noise_basis_points = 1e4 * sd, realized_five_minutes = round(simple_volatility(tsn, [:SPX]; fixed_spacing = 300)[:SPX] * sqrt(span), sigdigits = 4), paper_formula = round(sqrt(max(tsrv(noisy.logprice, K).TSRV, 0)), sigdigits = 4), two_scales_volatility = round(v[:SPX] * sqrt(span), sigdigits = 4), noise_estimate_basis_points = round(1e4 * sqrt(w[:SPX]), sigdigits = 3))
end

## Things to Try

- Make the noise correlated across ticks, for instance a moving average of the draws: the noise estimate and the correction are then only approximate.
- Thin the day to one tick every ten seconds and see how few ticks two scales needs.
- Run **Microstructure Noise** in High-Frequency Volatility on your own ticks, to see how much of one tick return is noise.